# IndicCrimLawLLM — Corpus Exploration (01)

**Purpose.** Visual sanity-check over the scraped Supreme Court criminal corpus, driven by the structured JSON snapshot produced by `scripts/inventory_corpus.py`. Re-run as the scraper progresses — the notebook reflects whatever the inventory most recently wrote.

**Inputs.** `data/processed/corpus_inventory.json` (regenerate via `python scripts/inventory_corpus.py`).

**What to look at first.** The "Concerning findings" section at the bottom — that's where we translate the plots into action items.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INVENTORY_PATH = PROJECT_ROOT / "data" / "processed" / "corpus_inventory.json"
MAPPING_PATH = PROJECT_ROOT / "data" / "mappings" / "ipc_bns_mapping.yaml"

inv = json.loads(INVENTORY_PATH.read_text(encoding="utf-8"))
print(f"Inventory timestamp: {inv['snapshot_at']}")
print(f"Corpus size:         {inv['corpus_size']:,} docs")
print(f"Runtime:             {inv['runtime_seconds']}s")

In [ ]:
# Headline numbers at a glance
ts = inv["text_stats"]["chars"]
cn = inv["citation_network"]
bn = inv["bench"]
st = inv["statutes"]
print(f"Median text length:              {ts['p50']:,} chars")
print(f"p90 text length:                 {ts['p90']:,} chars")
print(f"Huge docs (>200K chars):         {inv['text_stats']['huge_doc_count']}")
print(f"Tiny docs (<500 chars):          {inv['text_stats']['tiny_doc_count']}")
print(f"Avg cases cited per doc:         {cn['avg_cases_cited_per_doc']}")
print(f"% with internal citation:        {cn['docs_with_internal_citation_pct']}%")
print(f"Empty-bench docs:                {bn['empty_bench_pct']}%")
print(f"Top IPC mapping coverage (30):   {st['mapping_coverage']['top_ipc_in_mapping_count']}/30")
print(f"Near-duplicate pairs (J≥0.85):   {inv['duplicates']['pair_count']}")

In [ ]:
# Docs per year (scrape-year partition)
by_year = inv["by_year"]
df_year = pl.DataFrame({
    "year": list(by_year.keys()),
    "files": [v["files_on_disk"] for v in by_year.values()],
})
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_year["year"].to_list(), df_year["files"].to_list(), color="#3b82f6")
ax.set_title("Documents scraped per year")
ax.set_xlabel("scrape-year partition")
ax.set_ylabel("file count")
for i, v in enumerate(df_year["files"].to_list()):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Text length histogram (log-scale buckets pre-computed by the inventory script)
hist = inv["text_stats"]["char_histogram"]
df_hist = pl.DataFrame(hist)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(df_hist["bucket"].to_list(), df_hist["count"].to_list(), color="#10b981")
ax.set_yscale("log")
ax.set_title("Document text length distribution (chars, log y)")
ax.set_xlabel("char-count bucket")
ax.set_ylabel("doc count (log)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 acts cited (horizontal bar)
top_acts = inv["statutes"]["top_acts"][:20]
df_acts = pl.DataFrame(top_acts)
fig, ax = plt.subplots(figsize=(9, 6))
# Plot in descending order of count — reverse so largest sits on top
acts = df_acts["act"].to_list()[::-1]
counts = df_acts["count"].to_list()[::-1]
ax.barh(acts, counts, color="#6366f1")
ax.set_title("Top 20 acts cited in corpus")
ax.set_xlabel("citation count")
for i, v in enumerate(counts):
    ax.text(v, i, f" {v:,}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Top 30 IPC sections — coloured by whether they're in the IPC↔BNS mapping
import yaml

mapping_doc = yaml.safe_load(MAPPING_PATH.read_text(encoding="utf-8"))
mapped_ipc = {
    m["ipc"] for m in (mapping_doc.get("mappings") or []) if m.get("ipc")
}

top_ipc = inv["statutes"]["top_ipc"]
df_ipc = pl.DataFrame(top_ipc)
df_ipc = df_ipc.with_columns(
    pl.col("section").is_in(list(mapped_ipc)).alias("mapped"),
)

sections = df_ipc["section"].to_list()[::-1]
counts = df_ipc["count"].to_list()[::-1]
mapped = df_ipc["mapped"].to_list()[::-1]
colors = ["#10b981" if m else "#ef4444" for m in mapped]

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(sections, counts, color=colors)
ax.set_title("Top 30 IPC sections cited\n(green = in IPC↔BNS mapping, red = missing)")
ax.set_xlabel("citation count")
for i, v in enumerate(counts):
    ax.text(v, i, f" {v:,}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Bench-size distribution
dist = inv["bench"]["size_distribution"]
sizes = [int(k) for k in dist.keys()]
counts = list(dist.values())
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(s) for s in sizes], counts, color="#f59e0b")
ax.set_title("Bench-size distribution")
ax.set_xlabel("judges on bench")
ax.set_ylabel("doc count")
for i, v in enumerate(counts):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Language breakdown
lang = inv["language"]["counts"]
labels = list(lang.keys())
values = list(lang.values())
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(values, labels=labels, autopct="%1.1f%%", startangle=90,
       colors=sns.color_palette("pastel")[: len(labels)])
ax.set_title("Detected language of full_text")
plt.tight_layout()
plt.show()

## Concerning findings

The plots above render the numbers; these are the items that need follow-up work. Each finding states the evidence (numbers from the inventory snapshot), the downstream impact, and a recommended action. Revisit as the corpus grows — some findings (especially #4) will resolve as scale increases.

### Finding 1: IPC 34 (common intention) is a top-2 citation but unmapped

**Evidence.** IPC 34 is the 2nd most-cited IPC section in the corpus (`top_ipc` row 2, see above). It is **not** in `data/mappings/ipc_bns_mapping.yaml` — the `mapping_coverage` block lists `34` first in `top_ipc_missing_from_mapping`.

**Impact.** Virtually every multi-accused criminal judgment cites IPC 34 as the joint-liability hook. A model trained without an IPC 34 ↔ BNS analogue (BNS 3(5), the renumbered common-intention provision) will silently fail to reason about post-July-2024 charge-sheets. This is the single highest-leverage mapping gap.

**Recommended action.** Add IPC 34 → BNS 3(5) to the mapping as a Batch 3 priority, alongside the general-principles chapter (IPC 1-52, BNS 1-11). Follow with IPC 120A/B (already mapped) and IPC 149 (already mapped) to complete the joint-liability cluster.

### Finding 2: IPC 124A (sedition) cited but unmapped

**Evidence.** IPC 124A appears in `top_ipc_missing_from_mapping`. The mapping question for 124A is materially different from everything already in the table: the BNS renumbers/reshapes sedition into 150 ("acts endangering sovereignty, unity and integrity of India") with a modified actus reus, and some commentators argue the re-enactment *narrows* the sedition offence while others argue it *broadens* it.

**Impact.** This is a high-salience political-law section. Any legal question answered by the model about 124A ("is the Kedar Nath Singh test still good law under BNS 150?") will be wrong without an explicit, well-noted mapping entry. The transition is not clerical.

**Recommended action.** Add IPC 124A → BNS 150 as a `one_to_one` with `needs_verification: true` and an extended notes field capturing the scope-change debate; file as a scholarship observation in `bns_transition_findings.md`.

### Finding 3: Heavy right-tail in text length — 3 docs over 200K chars

**Evidence.** `huge_doc_count = 3`; `chars.max = 1.8M`; `chars.p99 = ~434K`. The p99 is already 14× the p50, and the tail is dominated by multi-judge constitutional-bench decisions (Kaushal Kishor, Sabarimala line, etc.).

**Impact.** A 1.8M-character doc is ~300K words — well beyond any model's context window for single-pass reasoning. Without a chunking strategy, training either silently truncates these (losing the ratio-decidendi sections) or crashes on memory. The same docs are also disproportionately influential in the citation network (see Finding 4), so dropping them is not acceptable either.

**Recommended action.** Before training corpus assembly, add a chunker that splits on judgment-structural markers ("JUDGMENT", "ORDER", para numbers) with overlap. Flag each mega-doc in the processed corpus with a `chunked: true` field so retrieval-time code knows it was split.

### Finding 4: 17 docs cite >50 other cases — training-signal concentration risk

**Evidence.** `suspicious_high_citation_docs` has 17 entries, topped by `66970168` (443 cases cited), `103640961` (317), `50602236` (197). These are 5-judge or 7-judge constitutional benches.

**Impact.** These ~20 judgments will appear disproportionately as the "cited" endpoint of the internal citation graph, giving them outsized influence on any citation-based training signal (retrieval-augmented training, citation-prediction tasks, graph embeddings). At the current 95-doc corpus size only 10.5% of docs have any internal citation — the signal is sparse, so concentration is maximally felt.

**Recommended action.** Monitor this as the corpus grows. Expect internal-citation density to climb toward 40-60% once we cross 5K docs; the concentration concern recedes once the long tail fills in. Flag any judgment cited by >1% of the eventual corpus for manual review (the landmark set).

### Finding 5: Citation-network sparsity at current scale

**Evidence.** Only 10.5% of docs cite at least one other doc that is also in our corpus. Average cases-cited per doc is 38, but most of those citations point to docs we haven't scraped yet.

**Impact.** Graph-based methods (citation embeddings, landmark detection, authority propagation) won't work yet. Any RAG eval that requires "retrieve cited case and check" will miss most targets. This is not a bug — it's the cold-start of a citation graph.

**Recommended action.** Do nothing now; revisit this number at 1K, 5K, and 20K doc milestones. If it doesn't cross ~40% at 20K docs, that suggests systematic under-scraping of the cited corpus (likely pre-2015 SC decisions) and argues for extending the scrape window backward.

## What's missing from IPC ↔ BNS mapping

Top-cited IPC sections in the corpus that are **not** yet in `data/mappings/ipc_bns_mapping.yaml`, ordered by citation frequency in the corpus. These are the highest-leverage additions for Batch 3.

| IPC Section | Corpus citations | Subject | BNS analogue (best guess, needs verification) |
|---|---|---|---|
| 34 | very high (top-2) | Common intention — vicarious liability of co-accused | BNS 3(5) |
| 201 | high | Causing disappearance of evidence of offence | BNS 238 |
| 161 | high | Public servant taking gratification | *Moved out of IPC in 1988 → Prevention of Corruption Act; cites survive in old judgments. No BNS analogue — already PCA territory.* |
| 193 | high | Punishment for false evidence | BNS 229 |
| 504 | high | Intentional insult intended to provoke breach of peace | BNS 352 |
| 295A | high | Deliberate and malicious acts intended to outrage religious feelings | BNS 299 |
| 124A | medium | Sedition — fiercely contested re-enactment | BNS 150 (narrower? broader? scholarship split) |
| 294 | medium | Obscene acts and songs in public | BNS 296 |

The IPC 161 row is a special case worth a scholarship note: citations to it in the corpus are almost entirely drawn from pre-1988 judgments or discussions of historical doctrine. The section was repealed from the IPC by the Prevention of Corruption Act, 1988. No BNS mapping is needed; the correct response is a `relationship: removed` entry with notes pointing to PCA. Handle as a Batch 3 side-task.

*(List regenerates automatically from `inv["statutes"]["mapping_coverage"]["top_ipc_missing_from_mapping"]` — re-run this notebook after each mapping-expansion commit and the green/red bars above will shift.)*